# 2. Algorytm genetyczny: dobór hiperparametrów

**Cel:** przeszukać dyskretną przestrzeń dwóch hiperparametrów i porównać liczbę ocen z poszukiwaniem losowym. Czas: 15 min. Wymaga `numpy`, `scikit-learn`.

**Uruchamianie:** wykonuj komórki kolejno. Dane są generowane lokalnie; nie jest potrzebny internet.

### Chromosom i budżet
Chromosom `(głębokość drzewa, minimalna liczba próbek w liściu)`. Oba algorytmy dostają po 15 wywołań oceny. Porównujemy także liczbę **unikalnych** kandydatów.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold,cross_val_score
X,y=make_moons(n_samples=240,noise=0.24,random_state=21)
cv=StratifiedKFold(3,shuffle=True,random_state=21)
depths=np.array([2,3,4,5,7,10]); leaves=np.array([1,2,4,8])
def evaluate(gene):
    d,l=map(int,gene)
    model=DecisionTreeClassifier(max_depth=int(depths[d]),min_samples_leaf=int(leaves[l]),random_state=21)
    return float(cross_val_score(model,X,y,cv=cv,scoring='accuracy').mean())


In [ ]:
rng=np.random.default_rng(21)
pop=[(int(rng.integers(6)),int(rng.integers(4))) for _ in range(5)]
visited=[]
for generation in range(3):
    scored=[(evaluate(g),g) for g in pop]
    visited.extend(scored)
    champion=max(scored)[1]
    parents=[g for _,g in sorted(scored,reverse=True)[:3]]
    pop=[champion]
    while len(pop)<5:
        a,b=rng.choice(3,size=2,replace=False)
        child=[parents[a][0],parents[b][1]]
        if rng.random()<0.35:
            k=int(rng.integers(2)); child[k]=int(rng.integers(6 if k==0 else 4))
        child[0]%=6; child[1]%=4
        pop.append(tuple(child))
best_score,best_gene=max(visited)
print('GA:',depths[best_gene[0]],leaves[best_gene[1]],round(best_score,3),'unikalne:',len({g for _,g in visited}))
random_visits=[(lambda g:(evaluate(g),g))((int(rng.integers(6)),int(rng.integers(4)))) for _ in range(15)]
r_score,r_gene=max(random_visits)
print('Losowo:',depths[r_gene[0]],leaves[r_gene[1]],round(r_score,3),'unikalne:',len({g for _,g in random_visits}))
assert len(visited)==len(random_visits)==15


**Analiza:** Powtórz z innym seedem. Przewaga pojedynczego przebiegu nie świadczy o przewadze metody; zbierz wyniki z wielu seedów. Czy powtarzanie ocen tego samego chromosomu jest sensownym kosztem?